In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [1]:
import kagglehub
path = kagglehub.dataset_download("akeshkumarhp/electronics-products-amazon-10k-items")

100%|██████████| 958k/958k [00:00<00:00, 98.1MB/s]

Extracting files...


In [6]:
file_path = r"/content/electronics_product.csv"
df= pd.read_csv(file_path)

In [7]:
df.drop(columns=['Unnamed: 0', 'image', 'link'], inplace=True)

In [8]:
df.head()

,name,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,4.0,965,"₹10,999","₹18,999"
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,4.3,"113,956","₹18,999","₹19,999"
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,4.2,"90,304","₹1,999","₹2,299"
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,4.1,"24,863","₹15,999","₹24,999"
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,4.3,"113,956","₹18,999","₹19,999"


In [9]:
import sys
# !{sys.executable} -m pip install sentence-transformers

Now that we have the library installed, let's load a pre-trained model from `sentence_transformers`. This model will help us convert our product names and search queries into meaningful numerical vectors (embeddings).

In [10]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
# 'all-MiniLM-L6-v2' is a good balance of speed and performance for many tasks
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Sentence Transformer model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer model loaded successfully.


Next, we will generate embeddings for all the product names in your `df` DataFrame. This might take a little while depending on the size of your dataset.

In [11]:
product_names = df['name'].tolist()
product_embeddings = model.encode(product_names, show_progress_bar=True)

print(f"Generated embeddings for {len(product_embeddings)} products.")
print(f"Shape of one embedding: {product_embeddings[0].shape}")

Batches:   0%|          | 0/300 [00:00<?, ?it/s]

Generated embeddings for 9600 products.
Shape of one embedding: (384,)


Now that we have the embeddings for all product names, we can create a search function. This function will take a search query, convert it into an embedding, and then find the most similar product embeddings using cosine similarity. The top N most similar products will be returned.

In [17]:
search_query = "laptop"
search_results = semantic_search(search_query, df, product_embeddings, model)
display(search_results)

,name,similarity_score
0,HP G8 Core Intel i3 11th Gen - (8 GB/512 GB SS...,0.604371
1,"HP 15s,12th Gen Intel Core i3-1215U, 15.6 inch...",0.597287
2,Lenovo IdeaPad Slim 3 Intel Celeron N4020 4th ...,0.592519
3,"Lenovo V15 Intel Celeron N4020 15.6"" (39.62 cm...",0.592489
4,(Renewed) Lenovo ThinkPad T450 Intel Core i5-5...,0.592232


In [16]:
search_query = "tv"
search_results = semantic_search(search_query, df, product_embeddings, model)
display(search_results)

,name,similarity_score
0,"Samsung 27-inch(68.58cm) M5 FHD Smart Monitor,...",0.483240
1,"Samsung 27-inch(68.58cm) M5 FHD Smart Monitor,...",0.483240
2,Samsung 32-Inch(80.13Cm) LED 1920 x 1080 Pixel...,0.476378
3,Rts&trade; High Speed 3D Full HD 1080p Support...,0.460950
4,Amazon Basics 21.5-Inch (54.5cm) LCD 1920 x 10...,0.454114


In [13]:
# Example usage:
search_query = "smartphone with good camera"
search_results = semantic_search(search_query, df, product_embeddings, model)
display(search_results)

,name,similarity_score
0,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.648994
1,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.640355
2,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.585849
3,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.580574
4,"Nokia G11 Android 12 Smartphone, Dual SIM, 3-D...",0.559233


In [14]:
# Example usage:
search_query = "smartphone with good camera"
search_results = semantic_search(search_query, df, product_embeddings, model)
display(search_results)

,name,similarity_score
0,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.648994
1,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.640355
2,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.585849
3,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.580574
4,"Nokia G11 Android 12 Smartphone, Dual SIM, 3-D...",0.559233


In [15]:
# Example usage:
search_query = "smartphone with good camera"
search_results = semantic_search(search_query, df, product_embeddings, model)
display(search_results)

,name,similarity_score
0,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.648994
1,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.640355
2,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.585849
3,"Nokia C01 Plus 4G, 5.45” HD+ Screen, Selfie Ca...",0.580574
4,"Nokia G11 Android 12 Smartphone, Dual SIM, 3-D...",0.559233


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, df, product_embeddings, model, top_n=5):
    # Encode the query
    query_embedding = model.encode([query])

    # Calculate cosine similarity between the query and all product embeddings
    similarities = cosine_similarity(query_embedding, product_embeddings)[0]

    # Get the indices of the top N most similar products
    top_n_indices = similarities.argsort()[-top_n:][::-1]

    # Retrieve the corresponding product names and their similarity scores
    results = []
    for i in top_n_indices:
        results.append({
            'name': df.iloc[i]['name'],
            'similarity_score': similarities[i]
        })
    return pd.DataFrame(results)

# Example usage:
# search_query = "smartphone with good camera"
# search_results = semantic_search(search_query, df, product_embeddings, model)
# display(search_results)